In [ ]:
# ============================================================
# 02 — BM25 RETRIEVAL (EB-NeRD)
# Lexical BM25 recall@K {50,100,200} over full pool. Content is weak on EB-NeRD.
# Fully self-contained EB-NeRD notebook. Hardcoded paths.
# ============================================================
!pip install lightgbm sentence-transformers polars -q
import os, glob, math, zipfile, numpy as np, polars as pl, datetime as dt, lightgbm as lgb, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
# ---- hardcoded EB-NeRD demo path (fast offline iteration) ----
DEMO = "/kaggle/input/datasets/donbosoc/ebnerd-small"
if not os.path.exists(f"{DEMO}/articles.parquet"):
    DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
print("DEMO:", DEMO)
PREFIX = "eb"
def pfx(x): return f"{PREFIX}:{x}"
def _prefix(col): return pl.concat_str([pl.lit(f"{PREFIX}:"), col.cast(pl.Utf8)])
def parse_split(base, split):
    """Parse EB-NeRD into unified schema: articles / impressions / history."""
    a = pl.read_parquet(f"{base}/articles.parquet")
    articles = a.select(
        article_id=_prefix(pl.col("article_id")),
        title=pl.col("title").fill_null(""),
        abstract=pl.col("subtitle").fill_null(""),
        body=pl.col("body").fill_null("") if "body" in a.columns else pl.lit(""),
        category=pl.col("category_str").fill_null(""),
        published_time=pl.col("published_time"),
    )
    b = pl.read_parquet(f"{base}/{split}/behaviors.parquet")
    cols = b.columns
    impressions = b.select(
        impression_id=pl.col("impression_id"),
        user_id=_prefix(pl.col("user_id")),
        timestamp=pl.col("impression_time"),
        candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
        labels=(pl.col("article_ids_clicked").list.eval(_prefix(pl.element()))
                if "article_ids_clicked" in cols else pl.lit(None)),
        session_id=(pl.col("session_id") if "session_id" in cols else pl.lit(0)),
    )
    h = pl.read_parquet(f"{base}/{split}/history.parquet")
    hist = {u: (arts or []) for u, arts in zip(
        h.select(_prefix(pl.col("user_id")))["user_id"].to_list(),
        h["article_id_fixed"].list.eval(_prefix(pl.element())).to_list())}
    return articles, impressions, hist


In [ ]:
import re
_WORD=re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
art,imp_tr,hist_tr = parse_split(DEMO,"train")
ids=art["article_id"].to_list()
titles=art["title"].to_list(); abstracts=art["abstract"].to_list()
corpus=[tok(f"{t} {a}") for t,a in zip(titles,abstracts)]
id_to_row={x:i for i,x in enumerate(ids)}
title_tok={ids[i]:tok(titles[i]) for i in range(len(ids))}
print("articles:",len(ids))


In [ ]:
class BM25:
    def __init__(s,c,k1=1.5,b=0.75):
        s.k1,s.b=k1,b;s.N=len(c);s.tf=[Counter(d) for d in c]
        s.dl=np.array([len(d) for d in c],float);s.avg=s.dl.mean() if s.N else 1.0
        df=Counter()
        for t in s.tf: df.update(t.keys())
        s.idf={w:math.log((s.N-d+.5)/(d+.5)+1) for w,d in df.items()}
    def scores_all(s,q):
        if not q: return np.zeros(s.N)
        out=np.zeros(s.N)
        for r in range(s.N):
            tf=s.tf[r];dn=s.k1*(1-s.b+s.b*s.dl[r]/s.avg);v=0.0
            for w in set(q):
                f=tf.get(w,0)
                if f: v+=s.idf.get(w,0)*(f*(s.k1+1))/(f+dn)
            out[r]=v
        return out
bm25=BM25(corpus); print("BM25 built")


In [ ]:
# recall@K: is the clicked article in top-K retrieved by BM25 over history-title query?
import random as _r; _r.seed(0)
eval_rows=[]
for row in imp_tr.iter_rows(named=True):
    labs=row["labels"] or []; uid=row["user_id"]
    if labs and hist_tr.get(uid):
        eval_rows.append((uid, labs[0]))
_r.shuffle(eval_rows); eval_rows=eval_rows[:2000]
print("eval impressions:", len(eval_rows))
Ks=[50,100,200]; hits={k:0 for k in Ks}; n=0
for uid,clicked in eval_rows:
    q=[]
    for x in hist_tr.get(uid,[])[-30:]: q.extend(title_tok.get(x,[]))
    sc=bm25.scores_all(q); order=np.argsort(-sc)
    cr=id_to_row.get(clicked)
    if cr is None: continue
    pos=np.where(order==cr)[0]
    if len(pos)==0: continue
    for k in Ks:
        if pos[0]<k: hits[k]+=1
    n+=1
print("\n=== EB-NeRD BM25 retrieval recall@K ===")
for k in Ks: print(f"  recall@{k}: {hits[k]/n:.4f}")
print("(content/lexical retrieval is weak on EB-NeRD ~0.03-0.04)")
